<a href="https://colab.research.google.com/github/MANIKANTH678/alml/blob/main/deepseeknewsolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Business entity resolution: S1 ↔ S2 / S3 (v2)

Matches each `source1` business to its records in `source2` and `source3`.
Run top to bottom (**Runtime → Run all**).

### What's new in v2 vs v1

| Area | v1 | v2 |
|---|---|---|
| Normalization | English-centric legal suffixes | Same list, but a broader filler set and a **French self-check** so unseen-country rows are handled correctly |
| Features | 15 features, 8 scorers | **+3 scorers (`QRatio`, name `partial`, addr `partial`) and 2 interaction features** |
| Model | `HistGradientBoostingClassifier` | **LightGBM** with early stopping (usually +1–3 F1 on this kind of pair-classification) |
| Blocking | lexical only | lexical + **optional embedding-based candidates** (multilingual MiniLM + FAISS), gated by `USE_EMBEDDING_BLOCKING` |
| Train/val split | random hash on S1 id | **stratified by `country × match-count bucket`** so `val` mirrors the mix of the training set |
| Recall check | yes | yes, plus per-country recall reported for `US`/`India` (and `France` if it appears in a future train set) |

The rest of the pipeline (DuckDB blocking, `rapidfuzz.process.cpdist`, streamed test featurization, cache on Drive) is unchanged.

**If a cell fails:** fix the cause and use *Run all* again rather than adding repair cells. The notebook stops by
itself if its normalization code or the Parquet cache is wrong, and it keeps its DuckDB database on local disk
(never put a `.duckdb` file on Google Drive; it gets corrupted there).

## 1. Setup

In [1]:
import sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'duckdb>=1.1', 'rapidfuzz>=3.6', 'anyascii', 'pyarrow',
                    'scikit-learn', 'lightgbm>=4.0'], check=True)
    if 'ER_USE_EMBEDDING_BLOCKING' in __import__('os').environ:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers>=2.5', 'faiss-cpu'], check=True)

In [2]:
import os, shutil, time
from contextlib import contextmanager
from pathlib import Path

import duckdb
import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
from anyascii import anyascii
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler
import lightgbm as lgb

# ---- paths (env vars let the same notebook run outside Colab) ----
DATA_ZIP  = Path(os.environ.get('ER_DATA_ZIP', '/content/drive/MyDrive/dataset.zip'))
DATA_DIR  = Path(os.environ.get('ER_DATA_DIR', '/content/dataset'))             # where the zip is extracted
CACHE_DIR = Path(os.environ.get('ER_CACHE_DIR',                                 # survives runtime resets
                 '/content/drive/MyDrive/er_cache' if IN_COLAB else 'er_cache'))
WORK_DIR  = Path(os.environ.get('ER_WORK_DIR', '/content/er_work' if IN_COLAB else 'er_work'))  # fast local disk

# ---- knobs ----
MAX_KEY_PAIRS    = 1_000    # drop a blocking key if (#S1 rows) × (#S2+S3 rows) sharing it exceeds this
MAX_CANDS_PER_S1 = 200      # keep at most this many candidates per S1 record (most shared keys first)
N_TRAIN_S1       = 200_000  # S1 entities used for training (all 2.2M are not needed)
N_VAL_S1         = 50_000   # held-out S1 entities for recall / threshold selection
MAX_TRAIN_ROWS   = 4_000_000  # negatives are subsampled above this
BETA             = 1.0      # set to the beta of the competition's F-beta metric
SEED             = 42
BATCH_ROWS       = 250_000  # candidate pairs featurized per batch
REBUILD          = False    # True: ignore cached normalized Parquet

# v2 additions
USE_EMBEDDING_BLOCKING = bool(int(os.environ.get('ER_USE_EMBEDDING_BLOCKING', '0')))
EMB_MODEL_NAME         = os.environ.get('ER_EMB_MODEL', 'paraphrase-multilingual-MiniLM-L12-v2')
EMB_TOP_K              = 30       # nearest S2/S3 neighbours per S1 record
EMB_BATCH              = 256

DUCKDB_MEMORY        = os.environ.get('ER_DUCKDB_MEMORY', '6GB')
DUCKDB_STREAM_MEMORY = os.environ.get('ER_DUCKDB_STREAM_MEMORY', '2GB')
DUCKDB_THREADS       = int(os.environ.get('ER_DUCKDB_THREADS', os.cpu_count() or 2))

# A DuckDB database file on Google Drive gets corrupted: keep it on local disk.
assert not str(WORK_DIR.resolve()).startswith('/content/drive'), 'WORK_DIR must be on local disk, not Google Drive'
WORK_DIR.mkdir(parents=True, exist_ok=True)


@contextmanager
def stage(name):
    t = time.time()
    print(f'▶ {name}')
    yield
    print(f'✔ {name}: {time.time() - t:,.1f}s')


def save_to_cache(local_path):
    """Copy a finished file from local disk to CACHE_DIR (Drive). A full Drive must not crash the run."""
    dest = CACHE_DIR / local_path.name
    try:
        shutil.copyfile(local_path, dest)
        return dest
    except OSError as e:
        dest.unlink(missing_ok=True)
        print(f'WARNING: could not save {local_path.name} to {CACHE_DIR}: {e}\n'
              f'  It is still available at {local_path} (lost when the runtime resets).\n'
              '  If Google Drive is full: delete er_checkpoint/ and other large files, then EMPTY THE DRIVE TRASH.')
        return local_path

In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

CACHE_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(CACHE_DIR).free / 2**30
print(f'free space for {CACHE_DIR}: {free_gb:,.1f} GB')
if free_gb < 5:
    print('WARNING: less than 5 GB free where the cache is written (it needs about 4 GB). On Google Drive, delete '
          'er_checkpoint/ and other large files, then empty the Drive trash.')

if not any(p for p in DATA_DIR.rglob('train_ground_truth.tsv') if '__MACOSX' not in p.parts):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with stage('unzip dataset'):
        subprocess.run(['unzip', '-q', '-o', str(DATA_ZIP), '-d', str(DATA_DIR)], check=True)

GT_PATH = next(p for p in DATA_DIR.rglob('train_ground_truth.tsv') if '__MACOSX' not in p.parts)
BASE = GT_PATH.parent.parent
SOURCES = {split: {s: BASE / split / f'{split}_source{s}.tsv' for s in (1, 2, 3)} for split in ('train', 'test')}
for split, files in SOURCES.items():
    for s, p in files.items():
        assert p.exists(), p
        print(f'{p.name:20s} {p.stat().st_size / 2**30:5.2f} GB')

readme = BASE.parent / 'README.md'
if readme.exists():
    print(readme.read_text()[:3000])

Mounted at /content/drive
free space for /content/drive/MyDrive/er_cache: 14.0 GB
▶ unzip dataset
✔ unzip dataset: 68.7s
train_source1.tsv     0.20 GB
train_source2.tsv     0.46 GB
train_source3.tsv     0.47 GB
test_source1.tsv      0.16 GB
test_source2.tsv      0.47 GB
test_source3.tsv      0.47 GB
# ML Challenge 2026 Problem Statement

## Business Entity Resolution Challenge

In large-scale commercial platforms, business identity data arrives from multiple independent sources — each contributing partial, noisy fragments of information about the same real-world entities. These fragments share no common identifiers, and the challenge of determining which records refer to the same business is known as Entity Resolution (ER). Your challenge is to build an ML solution that, given business records from 3 independent data sources with noisy and inconsistent fields, determines which records across sources refer to the same real-world business entity.

Source 1 is the deduplicated reference s

In [4]:
# The whole notebook uses this one connection. Do not replace it with a new duckdb.connect() in a later
# cell: that drops the to_ascii function, the spill directory and the memory limit set here.
con = duckdb.connect(str(WORK_DIR / 'er.duckdb'))
con.execute(f"SET temp_directory = '{WORK_DIR / 'spill'}'")
con.execute(f"SET memory_limit = '{DUCKDB_MEMORY}'")
con.execute(f'SET threads = {DUCKDB_THREADS}')
con.execute('SET preserve_insertion_order = false')
print('duckdb', duckdb.__version__, '| threads:', con.execute("SELECT current_setting('threads')").fetchone()[0])


def _to_ascii(col):
    # anyascii handles Devanagari/Tamil/Telugu/Bengali and accents: 'मॉडर्न फाइनेंस' -> 'modrn phainems'
    return pa.array([anyascii(s) if s is not None and not s.isascii() else s for s in col.to_pylist()],
                    type=pa.string())


con.create_function('to_ascii', _to_ascii, ['VARCHAR'], 'VARCHAR', type='arrow')

duckdb 1.3.2 | threads: 2


## 2. Normalize sources (cached as Parquet)

* `name_norm` – lower-case ASCII alphanumerics.
* `name_core` – `name_norm` without legal forms, honorifics and articles (a list that already covers
  the common French forms `SARL`, `SAS`, `SA`, `EURL`, `SCI`, `SNC` so unseen `France` rows are handled).
* `name_skel` – rough phonetic skeleton of `name_core`.
* `addr_norm` – address with street-type abbreviations expanded; `addr_nums` – sorted distinct numbers in it.

The self-check below now includes a French business name so a regression in the French handling fails in seconds.

In [6]:
LEGAL_AND_FILLER = [
    # legal forms (incl. common transliterations; French forms are included on purpose)
    'private', 'pvt', 'praivet', 'limited', 'ltd', 'limitad', 'llp', 'llc', 'l l c', 'inc', 'incorporated',
    'corp', 'corporation', 'co', 'company', 'plc', 'opc', 'sarl', 'sas', 'sasu', 'sa', 'eurl', 'sci', 'snc',
    'gmbh', 'pllc', 'lp', 'ltda', 'pte', 'pty', 'ag', 'nv', 'bv',
    # honorifics / prefixes
    'm s', 'shri', 'shree', 'sri', 'sree', 'shrii', 'smt', 'mr', 'mrs', 'ms', 'dr',
    # articles / connectors
    'the', 'and', 'of', 'et', 'le', 'la', 'les', 'de', 'du', 'des', 'aux', 'un', 'une',
]
ADDR_ABBR = {
    'rd': 'road', 'st': 'street', 'ave': 'avenue', 'av': 'avenue', 'blvd': 'boulevard', 'bd': 'boulevard',
    'dr': 'drive', 'ln': 'lane', 'ct': 'court', 'hwy': 'highway', 'pkwy': 'parkway', 'pl': 'place',
    'sq': 'square', 'ter': 'terrace', 'apt': 'apartment', 'ste': 'suite', 'fl': 'floor', 'bldg': 'building',
    'nagr': 'nagar', 'mkt': 'market', 'opp': 'opposite',
    'r': 'rue', 'che': 'chemin', 'chem': 'chemin', 'bd': 'boulevard',
}


def sql_clean(col):
    """SQL expression: lower-case ASCII tokens separated by single spaces."""
    ascii_ = (f"CASE WHEN regexp_matches(coalesce({col}, ''), '[^\\x00-\\x7F]') "
              f"THEN to_ascii({col}) ELSE coalesce({col}, '') END")
    return f"trim(regexp_replace(replace(lower({ascii_}), '&', ' and '), '[^a-z0-9]+', ' ', 'g'))"


def sql_squash(expr):
    return f"trim(regexp_replace({expr}, ' +', ' ', 'g'))"


STOP_RE = r'\b(' + '|'.join(LEGAL_AND_FILLER) + r')\b'


def sql_core(expr):
    return sql_squash(f"regexp_replace({expr}, '{STOP_RE}', ' ', 'g')")


def sql_skeleton(expr):
    e = f"regexp_replace({expr}, 'ph', 'f', 'g')"
    e = f"regexp_replace({e}, 'w', 'v', 'g')"
    e = f"regexp_replace({e}, 'm([kgcjtdsz])', 'n\\1', 'g')"
    return f"regexp_replace({e}, '\\B[aeiouy]', '', 'g')"


def sql_addr(expr):
    for short, full in ADDR_ABBR.items():
        expr = f"regexp_replace({expr}, '\\b{short}\\b', '{full}', 'g')"
    return expr


def build_recs(split):
    """Create table recs_<split> (one row per entity, all three sources), cached as Parquet."""
    pq = CACHE_DIR / f'recs_{split}.parquet'
    if pq.exists() and not REBUILD:
        con.execute(f"CREATE OR REPLACE TABLE recs_{split} AS SELECT * FROM read_parquet('{pq}')")
        return
    raw = ' UNION ALL '.join(
        f"SELECT {s}::TINYINT AS src, * FROM read_csv('{path}', delim='\\t', header=true, all_varchar=true, "
        f"quote='\"', escape='\"')"
        for s, path in SOURCES[split].items())
    con.execute(f"""
        CREATE OR REPLACE TABLE recs_{split} AS
        WITH raw AS ({raw}),
        n AS (
            SELECT src, trim(entity_id) AS entity_id,
                   lower(trim(coalesce(country, ''))) AS country,
                   business_name AS name_raw, business_address AS addr_raw,
                   regexp_matches(coalesce(business_name, ''), '[^\\x00-\\x7F]') AS non_ascii,
                   {sql_clean('business_name')} AS name_norm,
                   {sql_squash(sql_addr(sql_clean('business_address')))} AS addr_norm
            FROM raw
        ),
        c AS (SELECT *, {sql_core('name_norm')} AS name_core FROM n)
        SELECT *, {sql_skeleton('name_core')} AS name_skel,
               array_to_string(list_sort(list_distinct(regexp_extract_all(addr_norm, '[0-9]+'))), ' ') AS addr_nums
        FROM c
        QUALIFY row_number() OVER (PARTITION BY entity_id ORDER BY src) = 1
    """)
    got = dict(con.execute(f'SELECT src, count(*) FROM recs_{split} GROUP BY src').fetchall())
    for s, path in SOURCES[split].items():
        with open(path, 'rb') as f:
            lines = sum(1 for _ in f) - 1
        if got.get(s, 0) != lines:
            print(f'WARNING {path.name}: {lines:,} data lines but {got.get(s, 0):,} unique records loaded')
    local_pq = WORK_DIR / pq.name
    con.execute(f"COPY recs_{split} TO '{local_pq}' (FORMAT parquet)")
    if save_to_cache(local_pq) != local_pq:
        local_pq.unlink()


# ---- self-checks: fail in seconds instead of producing silently wrong data for hours ----
NAME_CASES = {  # raw name -> (name_core, name_skel, non_ascii)
    'M/S Producer SNK Infrastructure Pvt. Ltd.': ('producer snk infrastructure', 'prdcr snk infrstrctr', False),
    "Stephenie's Seafood": ('stephenie s seafood', 'stfn s sfd', False),
    'मॉडर्न फाइनेंस': ('modrn phainems', 'mdrn fnns', True),
    'Café de la Gare SARL': ('cafe gare', 'cf gr', True),
    'Shree Ram Marketing Private Limited': ('ram marketing', 'rm mrktng', False),
    'Société Générale SA': ('societe generale', 'sct gnrl', True),          # French, unseen in train
    'Boulangerie du Centre SAS': ('boulangerie centre', 'blngr cntr', False),  # French, unseen in train
}
ADDR_CASES = {  # raw address -> addr_norm
    '00719 Lincoln Ave, Carrollton, OH': '00719 lincoln avenue carrollton oh',
    'Plot No. 1, Opp. Bus Stand, Nehru Rd, Vadodara': 'plot no 1 opposite bus stand nehru road vadodara',
    # NOTE: `de`/`la`/`du`/etc. are only stripped from *names* (sql_core). Addresses keep them.
    '12 Rue de la Paix, Paris': '12 rue de la paix paris',
}
FIX_HINT = ('The normalization helpers in this notebook were modified (a common cause is doubled regex '
            'backslashes). Re-download the notebook instead of patching the SQL.')


def sql_values(strings):
    return 'VALUES ' + ', '.join("('" + s.replace("'", "''") + "')" for s in strings)


def check_normalizers():
    got = con.execute(f"""
        SELECT x, {sql_core(sql_clean('x'))}, {sql_skeleton(sql_core(sql_clean('x')))},
               regexp_matches(x, '[^\\x00-\\x7F]')
        FROM ({sql_values(NAME_CASES)}) t(x)
    """).fetchall()
    got += [(x, a) for x, a in con.execute(
        f"SELECT x, {sql_squash(sql_addr(sql_clean('x')))} FROM ({sql_values(ADDR_CASES)}) t(x)").fetchall()]
    expected = {**{k: (k, *v) for k, v in NAME_CASES.items()}, **{k: (k, v) for k, v in ADDR_CASES.items()}}
    bad = [f'  {row[0]!r}: got {row[1:]}, expected {expected[row[0]][1:]}' for row in got if row != expected[row[0]]]
    assert not bad, FIX_HINT + '\n' + '\n'.join(bad)
    print('✔ normalization self-check passed')


def check_recs(split, sample=20_000):
    """Recompute a sample of recs_<split> with the current helpers; catches a stale or broken Parquet cache."""
    bad = con.execute(f"""
        SELECT count(*) FILTER (WHERE name_norm <> {sql_clean('name_raw')}) AS name_norm,
               count(*) FILTER (WHERE name_core <> {sql_core('name_norm')}) AS name_core,
               count(*) FILTER (WHERE name_skel <> {sql_skeleton('name_core')}) AS name_skel,
               count(*) FILTER (WHERE addr_norm <> {sql_squash(sql_addr(sql_clean('addr_raw')))}) AS addr_norm,
               count(*) FILTER (WHERE non_ascii <> regexp_matches(coalesce(name_raw, ''), '[^\\x00-\\x7F]')) AS non_ascii
        FROM (SELECT * FROM recs_{split} USING SAMPLE {sample} ROWS)
    """).df()
    assert bad.to_numpy().sum() == 0, (
        f'recs_{split} does not match this notebook\'s normalization. '
        f'Delete {CACHE_DIR / f"recs_{split}.parquet"} or set REBUILD = True, then rerun.\n{bad.to_string(index=False)}')
    print(f'✔ recs_{split} matches the normalization code')


check_normalizers()

✔ normalization self-check passed


In [7]:
with stage('normalize train'):
    build_recs('train')
with stage('normalize test'):
    build_recs('test')

for split in ('train', 'test'):
    check_recs(split)
    print(split, con.execute(f'SELECT src, count(*) FROM recs_{split} GROUP BY src ORDER BY src').fetchall())
con.sql("""SELECT src, name_raw, name_norm, name_core, name_skel, addr_norm, addr_nums
           FROM recs_train USING SAMPLE 8 ROWS""").df()

▶ normalize train


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✔ normalize train: 569.6s
▶ normalize test


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✔ recs_train matches the normalization code
train [(1, 2206821), (2, 5034616), (3, 5285603)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✔ recs_test matches the normalization code
test [(1, 1732544), (2, 4887273), (3, 5082316)]


,src,name_raw,name_norm,name_core,name_skel,addr_norm,addr_nums
0,1,Tirupati Logistics Pvt Ltd,tirupati logistics pvt ltd,tirupati logistics,trpt lgstcs,b gopala gowda near anjaneya tempal subhash ro...,None
1,1,Super Trading Private Limited,super trading private limited,super trading,spr trdng,10 g f mcd park marg hamkunt colony near mcd p...,10
2,2,अर्बन कंसल्टेंसी प्राइवेट लिमिटेड,arbn kmsltemsi praivet limited,arbn kmsltemsi,arbn knsltns,f 196 kh no 98 gali no 11 khajoori khas delhi ...,11 196 98
3,2,Sky Management Hotel Private Limited,sky management hotel private limited,sky management hotel,sk mngmnt htl,pscimbng 13 lyons range turner morrison buildi...,13 15 3
4,2,A F & M MIDA INC,a f and m mida inc,a f m mida,a f m md,washington county ok 1690 sunset boulevard,1690
5,2,Shivam Media Private-Limited,shivam media private limited,shivam media,shvm md,new house no c 1 19 ff old house no 7 kh no 78...,1 16 19 675 7 78
6,2,Prime Gores Partners,prime gores partners,prime gores partners,prm grs prtnrs,216 national drive murfreesboro tn,216
7,2,Siva Industrial Private Ltd,siva industrial private ltd,siva industrial,sv indstrl,603 wz 60 a ground floor chaukhandi extn tilak...,60 603


## 3. Blocking

Identical to v1: for every record we pick the two **rarest** tokens of `name_core` and of the address (by document
frequency within the country), and build keys from them (`n1`, `n2`, `px`, `sk`, `a2`, `na`). Keys shared by
more than `MAX_KEY_PAIRS` records are dropped.

**v2 addition:** if `USE_EMBEDDING_BLOCKING=1`, a multilingual sentence-transformer also contributes candidates
via FAISS top-`EMB_K` neighbours per S1 record. This helps recall when two records share neither a rare name
token nor a rare address token (common for unseen-country rows and for heavy transliterations).

In [8]:
def rare_tokens(split, text_col, out, min_len):
    """out(entity_id, r1, r2): the two rarest tokens of text_col (ties broken alphabetically)."""
    tokens = f"""
        SELECT e.eid, e.country, unnest(string_split(r.{text_col}, ' ')) AS tok
        FROM recs_{split} r JOIN ents_{split} e USING (entity_id)"""
    token_filter = f"length(tok) >= {min_len} AND NOT regexp_matches(tok, '^[0-9]+$')"
    shift = 2 ** 31  # key = df * shift + tid orders by document frequency, then alphabetically
    con.execute(f"""
        CREATE OR REPLACE TABLE _rt_dict AS
        SELECT country, tok, (row_number() OVER (ORDER BY tok, country))::INTEGER AS tid
        FROM (SELECT DISTINCT country, tok FROM ({tokens}) WHERE {token_filter})
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE _rt_et AS
        SELECT DISTINCT t.eid, d.tid FROM ({tokens}) t JOIN _rt_dict d USING (country, tok)
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE _rt_key AS
        WITH df AS (SELECT tid, count(*) AS df FROM _rt_et GROUP BY tid)
        SELECT et.eid, df.df * {shift} + et.tid AS key FROM _rt_et et JOIN df USING (tid)
    """)
    con.execute('CREATE OR REPLACE TABLE _rt_m1 AS SELECT eid, min(key) AS k1 FROM _rt_key GROUP BY eid')
    con.execute("""
        CREATE OR REPLACE TABLE _rt_m2 AS
        SELECT k.eid, min(k.key) AS k2 FROM _rt_key k JOIN _rt_m1 m USING (eid) WHERE k.key > m.k1 GROUP BY k.eid
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE {out} AS
        SELECT e.entity_id, d1.tok AS r1, d2.tok AS r2
        FROM _rt_m1 m1
        JOIN ents_{split} e USING (eid)
        JOIN _rt_dict d1 ON d1.tid = m1.k1 % {shift}
        LEFT JOIN _rt_m2 m2 USING (eid)
        LEFT JOIN _rt_dict d2 ON d2.tid = m2.k2 % {shift}
    """)
    for t in ('_rt_dict', '_rt_et', '_rt_key', '_rt_m1', '_rt_m2'):
        con.execute(f'DROP TABLE {t}')


KEY_TYPES = ['n1', 'n2', 'px', 'sk', 'a2', 'na']


def build_keys(split):
    con.execute(f'''CREATE OR REPLACE TABLE ents_{split} AS
                    SELECT (row_number() OVER ())::INTEGER AS eid, entity_id, country FROM recs_{split}''')
    with stage(f'rarest name tokens ({split})'):
        rare_tokens(split, 'name_core', f'nrare_{split}', 2)
    with stage(f'rarest address tokens ({split})'):
        rare_tokens(split, 'addr_norm', f'arare_{split}', 3)
    t = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE keys_{split} AS
        WITH b AS (
            SELECT r.entity_id, r.src, r.country, replace(r.name_core, ' ', '') AS core_ns,
                   replace(r.name_skel, ' ', '') AS skel_ns, n.r1 AS n1, n.r2 AS n2, a.r1 AS a1, a.r2 AS a2
            FROM recs_{split} r
            LEFT JOIN nrare_{split} n USING (entity_id)
            LEFT JOIN arare_{split} a USING (entity_id)
        )
        SELECT * FROM (
            SELECT entity_id, src, 'n1' AS kt, 'n1|' || country || '|' || n AS k
            FROM (SELECT *, unnest([n1, n2]) AS n FROM b) WHERE n IS NOT NULL
            UNION ALL
            SELECT entity_id, src, 'n2', 'n2|' || country || '|' || least(n1, n2) || ' ' || greatest(n1, n2)
            FROM b WHERE n2 IS NOT NULL
            UNION ALL
            SELECT entity_id, src, 'px', 'px|' || country || '|' || left(core_ns, 6) FROM b WHERE length(core_ns) >= 4
            UNION ALL
            SELECT entity_id, src, 'sk', 'sk|' || country || '|' || left(skel_ns, 8) FROM b WHERE length(skel_ns) >= 3
            UNION ALL
            SELECT entity_id, src, 'a2', 'a2|' || country || '|' || least(a1, a2) || ' ' || greatest(a1, a2)
            FROM b WHERE a2 IS NOT NULL
            UNION ALL
            SELECT entity_id, src, 'na', 'na|' || country || '|' || n || '|' || a
            FROM (SELECT entity_id, src, country, n, unnest([a1, a2]) AS a
                  FROM (SELECT *, unnest([n1, n2]) AS n FROM b))
            WHERE n IS NOT NULL AND a IS NOT NULL
        )
    """)
    con.execute(f"""
        CREATE OR REPLACE TABLE blocks_{split} AS
        SELECT k, any_value(kt) AS kt,
               count(*) FILTER (WHERE src = 1) AS n1, count(*) FILTER (WHERE src > 1) AS n23
        FROM keys_{split} GROUP BY k HAVING n1 > 0 AND n23 > 0
    """)
    n_keys = con.execute(f'SELECT count(*) FROM keys_{split}').fetchone()[0]
    print(f'  keys_{split}: {n_keys:,} rows, built with block stats in {time.time() - t:,.1f}s')
    return con.sql(f"""
        SELECT kt, count(*) AS keys,
               count(*) FILTER (WHERE n1 * n23 <= {MAX_KEY_PAIRS}) AS keys_kept,
               sum(n1 * n23) FILTER (WHERE n1 * n23 <= {MAX_KEY_PAIRS}) AS pairs_kept,
               sum(n1 * n23) FILTER (WHERE n1 * n23 > {MAX_KEY_PAIRS}) AS pairs_dropped,
               max(n1 * n23) AS largest_block
        FROM blocks_{split} GROUP BY kt ORDER BY kt
    """).df()


def build_candidates(split, out, left_ids=None):
    """out(id1, id2, n_keys, min_block, k_*): S1 × (S2 ∪ S3) pairs sharing at least one kept key."""
    left_filter = f'JOIN {left_ids} USING (entity_id)' if left_ids else ''
    flags = ', '.join(f"max((l.kt = '{kt}')::TINYINT) AS k_{kt}" for kt in KEY_TYPES)
    con.execute(f"""
        CREATE OR REPLACE TABLE {out} AS
        WITH b AS (SELECT k, n1 * n23 AS bp FROM blocks_{split} WHERE n1 * n23 <= {MAX_KEY_PAIRS}),
             l AS (SELECT entity_id, kt, k FROM keys_{split} {left_filter} WHERE src = 1),
             r AS (SELECT entity_id, k FROM keys_{split} WHERE src > 1)
        SELECT l.entity_id AS id1, r.entity_id AS id2,
               count(*) AS n_keys, min(b.bp) AS min_block,
               {flags}
        FROM l JOIN b USING (k) JOIN r USING (k)
        GROUP BY l.entity_id, r.entity_id
        QUALIFY row_number() OVER (PARTITION BY id1 ORDER BY count(*) DESC, min(b.bp), id2) <= {MAX_CANDS_PER_S1}
    """)
    n, n1 = con.execute(f'SELECT count(*), count(DISTINCT id1) FROM {out}').fetchone()
    print(f'{out}: {n:,} pairs for {n1:,} S1 records')


with stage('blocking keys (train)'):
    display(build_keys('train'))

▶ blocking keys (train)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

▶ rarest name tokens (train)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  keys_train: 120,189,169 rows, built with block stats in 346.2s


,kt,keys,keys_kept,pairs_kept,pairs_dropped,largest_block
0,a2,1251459,1246540,19529241.0,8.863158e+07,29724600
1,n1,116420,91796,10495517.0,3.674770e+10,1567463352
2,n2,993510,982417,13367876.0,4.215349e+08,70513091
3,na,6287149,6281636,49811214.0,2.164742e+07,422894
4,px,334234,319341,11545817.0,9.699223e+09,987020064
5,sk,931795,920154,15856564.0,1.318238e+09,396515754


✔ blocking keys (train): 632.9s


### Optional: embedding-based candidates (v2)

Set `USE_EMBEDDING_BLOCKING=1` (or `ER_USE_EMBEDDING_BLOCKING=1` in the environment) to add FAISS top-`k`
candidates from a multilingual sentence-transformer. This costs ~2–5 minutes of extra encoding per split on a
Colab CPU runtime, and the model file itself is ~470 MB. Skip it if you are tight on time or memory.

In [9]:
def build_embedding_candidates(split, out):
    """Union the lexical candidates with FAISS nearest neighbours from a multilingual encoder."""
    import faiss
    from sentence_transformers import SentenceTransformer

    model_emb = SentenceTransformer(EMB_MODEL_NAME)
    dim = model_emb.get_sentence_embedding_dimension()
    print(f'embedding model {EMB_MODEL_NAME} ({dim}-dim)')

    def encode(src_filter):
        df = con.sql(f"""
            SELECT entity_id, coalesce(name_core, '') || ' | ' || coalesce(addr_norm, '') AS txt
            FROM recs_{split} WHERE {src_filter} ORDER BY entity_id
        """).df()
        emb = model_emb.encode(df['txt'].tolist(), batch_size=EMB_BATCH, show_progress_bar=True,
                               normalize_embeddings=True, convert_to_numpy=True)
        return df['entity_id'].tolist(), emb.astype('float32')

    s1_ids, s1_emb = encode('src = 1')
    s23_ids, s23_emb = encode('src > 1')

    index = faiss.IndexFlatIP(dim)
    index.add(s23_emb)
    _, nn = index.search(s1_emb, EMB_TOP_K)

    rows = []
    for i, neigh in enumerate(nn):
        id1 = s1_ids[i]
        for j in neigh:
            if j >= 0:
                rows.append((id1, s23_ids[j]))
    emb_pairs = pd.DataFrame(rows, columns=['id1', 'id2'])
    con.register('emb_pairs_view', emb_pairs)
    con.execute(f"""
        CREATE OR REPLACE TABLE emb_cands_{split} AS
        SELECT p.id1, p.id2, 1 AS n_keys, 1 AS min_block,
               {' , '.join(f"0::TINYINT AS k_{kt}" for kt in KEY_TYPES)}, 0 AS src2
        FROM emb_pairs_view p
    """)  # src2 is filled in by the featurizer from recs_ anyway
    con.unregister('emb_pairs_view')
    n = con.execute(f'SELECT count(*) FROM emb_cands_{split}').fetchone()[0]
    print(f'emb_cands_{split}: {n:,} pairs')
    del model_emb, s1_emb, s23_emb


def merge_emb_candidates(split, out):
    """Combine lexical + embedding candidates and keep the top MAX_CANDS_PER_S1 per S1."""
    con.execute(f"""
        CREATE OR REPLACE TABLE {out} AS
        WITH u AS (
            SELECT id1, id2, max(n_keys) AS n_keys, min(min_block) AS min_block,
                   {' , '.join(f'max(k_{kt}) AS k_{kt}' for kt in KEY_TYPES)}
            FROM (
                SELECT * FROM {out}
                UNION ALL
                SELECT * FROM emb_cands_{split}
            )
            GROUP BY id1, id2
        )
        SELECT * FROM u
        QUALIFY row_number() OVER (PARTITION BY id1 ORDER BY n_keys DESC, min_block, id2) <= {MAX_CANDS_PER_S1}
    """)
    n, n1 = con.execute(f'SELECT count(*), count(DISTINCT id1) FROM {out}').fetchone()
    print(f'{out} (merged with embeddings): {n:,} pairs for {n1:,} S1 records')


print('USE_EMBEDDING_BLOCKING =', USE_EMBEDDING_BLOCKING)

USE_EMBEDDING_BLOCKING = False


### Train / validation split of S1 entities and blocking recall (v2: stratified)

The split is **stratified by `country × match-count bucket`** so `val` mirrors the mix of the training set
(it always contains positive, single-match and many-match S1 entities from every country).

Recall = share of ground-truth pairs of the validation entities that survive blocking, reported per country.

In [10]:
con.execute(f"""
    CREATE OR REPLACE TABLE gt AS
    SELECT DISTINCT trim(source1_entity_id) AS id1, trim(m) AS id2
    FROM (SELECT source1_entity_id, unnest(string_split(matched_entity_ids, ',')) AS m
          FROM read_csv('{GT_PATH}', delim='\\t', header=true, all_varchar=true))
    WHERE trim(m) <> ''
""")
n_s1 = con.execute('SELECT count(*) FROM recs_train WHERE src = 1').fetchone()[0]
n_val = min(N_VAL_S1, n_s1 // 5)
n_train = min(N_TRAIN_S1, n_s1 - n_val)

# Stratified split: within each (country, match-count bucket), take ~20% as val, then subsample to the caps.
con.execute(f"""
    CREATE OR REPLACE TABLE s1_split AS
    WITH base AS (
        SELECT r.entity_id, r.country,
               LEAST(coalesce(cnt.n_matches, 0), 5) AS bucket
        FROM recs_train r
        LEFT JOIN (SELECT id1, count(*) AS n_matches FROM gt GROUP BY id1) cnt ON cnt.id1 = r.entity_id
        WHERE r.src = 1
    ),
    ranked AS (
        SELECT entity_id,
               row_number() OVER (PARTITION BY country, bucket ORDER BY hash(entity_id || '{SEED}')) AS rn,
               count(*)      OVER (PARTITION BY country, bucket) AS grp_size
        FROM base
    ),
    assigned AS (
        SELECT entity_id,
               CASE WHEN rn <= GREATEST(1, grp_size // 5) THEN 'val' ELSE 'train' END AS part
        FROM ranked
    ),
    subsampled AS (
        SELECT entity_id, part,
               row_number() OVER (PARTITION BY part ORDER BY hash(entity_id || 'sub{SEED}')) AS rn
        FROM assigned
    )
    SELECT entity_id, part FROM subsampled
    WHERE (part = 'val'   AND rn <= {n_val})
       OR (part = 'train' AND rn <= {n_train})
""")
con.execute("CREATE OR REPLACE TABLE ids_train AS SELECT entity_id FROM s1_split WHERE part = 'train'")
con.execute("CREATE OR REPLACE TABLE ids_val   AS SELECT entity_id FROM s1_split WHERE part = 'val'")
print(f'gt pairs: {con.execute("SELECT count(*) FROM gt").fetchone()[0]:,} | '
      f'train S1: {con.execute("SELECT count(*) FROM ids_train").fetchone()[0]:,} | '
      f'val S1: {con.execute("SELECT count(*) FROM ids_val").fetchone()[0]:,}')

with stage('candidates (train + val)'):
    build_candidates('train', 'cands_train', 'ids_train')
    build_candidates('train', 'cands_val', 'ids_val')
    if USE_EMBEDDING_BLOCKING:
        with stage('embedding candidates (train)'):
            build_embedding_candidates('train', 'cands_train')
            build_embedding_candidates('train', 'cands_val')
            merge_emb_candidates('train', 'cands_train')
            merge_emb_candidates('train', 'cands_val')

via = ', '.join(f'round(sum(c.k_{kt}) / count(*), 4) AS via_{kt}' for kt in KEY_TYPES)
recall = con.sql(f"""
    SELECT count(*) AS gt_pairs, count(c.id1) AS found, round(count(c.id1) / count(*), 4) AS blocking_recall, {via}
    FROM gt JOIN ids_val v ON gt.id1 = v.entity_id
    LEFT JOIN cands_val c ON c.id1 = gt.id1 AND c.id2 = gt.id2
""").df()
display(recall)

# Per-country recall — spot France if it ever appears in a future train set
per_country = con.sql("""
    SELECT r.country,
           count(*) AS gt_pairs,
           count(c.id1) AS found,
           round(count(c.id1) / count(*), 4) AS blocking_recall
    FROM gt JOIN ids_val v ON gt.id1 = v.entity_id
            JOIN recs_train r ON r.entity_id = gt.id1
            LEFT JOIN cands_val c ON c.id1 = gt.id1 AND c.id2 = gt.id2
    GROUP BY r.country ORDER BY r.country
""").df()
display(per_country)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

gt pairs: 7,638,365 | train S1: 200,000 | val S1: 50,000
▶ candidates (train + val)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cands_train: 7,067,572 pairs for 199,763 S1 records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cands_val: 1,771,383 pairs for 49,947 S1 records
✔ candidates (train + val): 78.8s


,gt_pairs,found,blocking_recall,via_n1,via_n2,via_px,via_sk,via_a2,via_na
0,173568,159799,0.9207,0.1332,0.4462,0.2375,0.4631,0.6344,0.7592


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,country,gt_pairs,found,blocking_recall
0,india,69431,60176,0.8667
1,us,104137,99623,0.9567


## 4. Features and training

Feature set in v2 adds three more string scorers (`QRatio` on the core name, `partial_ratio` on the whole name
and on the address) plus two interaction terms (`name_tset × addr_tset`, `core_tsort × skel_ratio`). The model
is **LightGBM** with early stopping on a 10% slice of the training candidates.

In [11]:
FEATURE_SQL = """
    SELECT c.*, {label} AS label,
           a.name_norm AS n1, b.name_norm AS n2, a.name_core AS c1, b.name_core AS c2,
           a.name_skel AS sk1, b.name_skel AS sk2, a.addr_norm AS a1, b.addr_norm AS a2,
           a.addr_nums AS d1, b.addr_nums AS d2, b.src AS src2, (a.non_ascii OR b.non_ascii) AS non_ascii
    FROM {cands} c
    JOIN recs_{split} a ON a.entity_id = c.id1
    JOIN recs_{split} b ON b.entity_id = c.id2
    {gt_join}
    {where}
"""

FEATURES = [
    # v1 string similarities
    'name_ratio', 'name_tset',
    'core_tsort', 'core_partial', 'core_jw',
    'skel_ratio',
    'addr_tset', 'addr_tsort',
    'num_tset',
    # v2 string similarities
    'name_partial', 'core_qratio', 'addr_partial',
    # derived numerics
    'num_missing', 'core_empty', 'len_ratio',
    'n_keys', 'log_min_block',
    *[f'k_{kt}' for kt in KEY_TYPES],
    'src2', 'non_ascii',
    # interaction features
    'name_addr_interaction', 'core_skel_interaction',
]


def featurize(df):
    def sim(scorer, x, y):
        return process.cpdist(df[x].tolist(), df[y].tolist(), scorer=scorer, workers=-1)

    out = pd.DataFrame({
        'name_ratio':   sim(fuzz.ratio, 'n1', 'n2'),
        'name_tset':    sim(fuzz.token_set_ratio, 'n1', 'n2'),
        'name_partial': sim(fuzz.partial_ratio, 'n1', 'n2'),
        'core_tsort':   sim(fuzz.token_sort_ratio, 'c1', 'c2'),
        'core_partial': sim(fuzz.partial_ratio, 'c1', 'c2'),
        'core_qratio':  sim(fuzz.QRatio, 'c1', 'c2'),
        'core_jw':      sim(JaroWinkler.normalized_similarity, 'c1', 'c2'),
        'skel_ratio':   sim(fuzz.ratio, 'sk1', 'sk2'),
        'addr_tset':    sim(fuzz.token_set_ratio, 'a1', 'a2'),
        'addr_tsort':   sim(fuzz.token_sort_ratio, 'a1', 'a2'),
        'addr_partial': sim(fuzz.partial_ratio, 'a1', 'a2'),
        'num_tset':     sim(fuzz.token_set_ratio, 'd1', 'd2'),
    }, index=df.index)
    l1, l2 = df['c1'].str.len(), df['c2'].str.len()
    out['num_missing'] = (df['d1'] == '').astype(np.int8) + (df['d2'] == '').astype(np.int8)
    out['core_empty'] = ((l1 == 0) | (l2 == 0)).astype(np.int8)
    out['len_ratio'] = (np.minimum(l1, l2) / np.maximum(np.maximum(l1, l2), 1)).astype(np.float32)
    out['log_min_block'] = np.log1p(df['min_block']).astype(np.float32)
    for col in ['n_keys', *[f'k_{kt}' for kt in KEY_TYPES], 'src2']:
        out[col] = df[col].astype(np.int16)
    out['non_ascii'] = df['non_ascii'].astype(np.int8)
    out['name_addr_interaction'] = (out['name_tset'] * out['addr_tset']).astype(np.float32)
    out['core_skel_interaction'] = (out['core_tsort'] * out['skel_ratio']).astype(np.float32)
    return out[FEATURES]


def stream_features(cands, split, with_label, neg_rate=1.0):
    """Yield (Arrow batch with id1/id2/label, feature frame) in batches of BATCH_ROWS pairs.
    neg_rate < 1 keeps that (deterministic) fraction of negatives, before any feature is computed."""
    where = (f"WHERE g.id1 IS NOT NULL OR hash(c.id1 || c.id2 || '{SEED}') % 1000000 < {int(neg_rate * 1e6)}"
             if neg_rate < 1 else '')
    sql = FEATURE_SQL.format(
        cands=cands, split=split, where=where,
        label='(g.id1 IS NOT NULL)' if with_label else 'NULL::BOOLEAN',
        gt_join='LEFT JOIN gt g ON g.id1 = c.id1 AND g.id2 = c.id2' if with_label else '')
    con.execute(f"SET memory_limit = '{DUCKDB_STREAM_MEMORY}'")
    try:
        res = con.execute(sql)
        reader = (res.to_arrow_reader if hasattr(res, 'to_arrow_reader') else res.fetch_record_batch)(BATCH_ROWS)
        for batch in reader:
            yield batch, featurize(batch.to_pandas())
    finally:
        con.execute(f"SET memory_limit = '{DUCKDB_MEMORY}'")


def load_features(cands, split, neg_rate=1.0):
    X, y = [], []
    for batch, feats in stream_features(cands, split, with_label=True, neg_rate=neg_rate):
        X.append(feats)
        y.append(batch.column('label').to_numpy(zero_copy_only=False).astype(bool))
    return pd.concat(X, ignore_index=True), np.concatenate(y)


n_all, n_pos = con.execute("""
    SELECT count(*), count(g.id1) FROM cands_train c LEFT JOIN gt g ON g.id1 = c.id1 AND g.id2 = c.id2
""").fetchone()
neg_rate = min(1.0, max(MAX_TRAIN_ROWS - n_pos, 0) / max(n_all - n_pos, 1))
print(f'train candidates: {n_all:,}  positives: {n_pos:,}  negatives kept: {neg_rate:.1%}')

with stage('features (train)'):
    X_tr, y_tr = load_features('cands_train', 'train', neg_rate)
print(f'training rows: {len(y_tr):,}')

with stage('train model (LightGBM)'):
    from sklearn.model_selection import train_test_split
    X_fit, X_es, y_fit, y_es = train_test_split(X_tr, y_tr, test_size=0.1, random_state=SEED, stratify=y_tr)
    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        X_fit, y_fit,
        eval_set=[(X_es, y_es)],
        eval_metric='binary_logloss',
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False),
                   lgb.log_evaluation(period=100)],
    )
print('best iteration:', model.best_iteration_)
del X_tr, y_tr, X_fit, X_es, y_fit, y_es

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

train candidates: 7,067,572  positives: 638,176  negatives kept: 52.3%
▶ features (train)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✔ features (train): 196.6s
training rows: 3,998,887
▶ train model (LightGBM)
[100]	valid_0's binary_logloss: 0.0328297
[200]	valid_0's binary_logloss: 0.0272322
[300]	valid_0's binary_logloss: 0.0256345
[400]	valid_0's binary_logloss: 0.0248106
[500]	valid_0's binary_logloss: 0.0241624
[600]	valid_0's binary_logloss: 0.0237121
[700]	valid_0's binary_logloss: 0.0233928
[800]	valid_0's binary_logloss: 0.0231259
[900]	valid_0's binary_logloss: 0.0228707
[1000]	valid_0's binary_logloss: 0.0226541
[1100]	valid_0's binary_logloss: 0.0224868
[1200]	valid_0's binary_logloss: 0.0223243
[1300]	valid_0's binary_logloss: 0.0221827
[1400]	valid_0's binary_logloss: 0.0220679
[1500]	valid_0's binary_logloss: 0.0219834
[1600]	valid_0's binary_logloss: 0.0218914
[1700]	valid_0's binary_logloss: 0.0218021
[1800]	valid_0's binary_logloss: 0.02172
[1900]	valid_0's binary_logloss: 0.021648
[2000]	valid_0's binary_logloss: 0.0215869
✔ train model (LightGBM): 1,268.5s
best iteration: 1997


## 5. Validation and threshold

Pair-level F-beta on the held-out S1 entities. Ground-truth pairs that blocking never produced count as false
negatives, so this is an honest end-to-end estimate.

In [12]:
with stage('features + predict (val)'):
    X_val, y_val = load_features('cands_val', 'train')
    p_val = model.predict_proba(X_val)[:, 1]
total_pos = con.execute('SELECT count(*) FROM gt JOIN ids_val v ON gt.id1 = v.entity_id').fetchone()[0]

order = np.argsort(-p_val)
tp = np.cumsum(y_val[order])
fp = np.cumsum(~y_val[order])
fn = total_pos - tp
b2 = BETA ** 2
fbeta = (1 + b2) * tp / np.maximum((1 + b2) * tp + b2 * fn + fp, 1)
best = int(np.argmax(fbeta))
THRESHOLD = float(p_val[order][best])
print(f'best F{BETA:g} = {fbeta[best]:.4f} at threshold {THRESHOLD:.4f}')
print(f'precision = {tp[best] / (tp[best] + fp[best]):.4f}  recall = {tp[best] / total_pos:.4f} '
      f'(blocking recall ceiling = {y_val.sum() / total_pos:.4f})')

joblib.dump({'model': model, 'threshold': THRESHOLD, 'features': FEATURES,
             'max_key_pairs': MAX_KEY_PAIRS, 'max_cands_per_s1': MAX_CANDS_PER_S1,
             'use_embedding_blocking': USE_EMBEDDING_BLOCKING},
            WORK_DIR / 'er_model.joblib')
save_to_cache(WORK_DIR / 'er_model.joblib')
del X_val, y_val, p_val

▶ features + predict (val)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✔ features + predict (val): 530.2s
best F1 = 0.9242 at threshold 0.6434
precision = 0.9692  recall = 0.8832 (blocking recall ceiling = 0.9207)


## 6. Predict on test

Features are computed and scored batch by batch. Output uses the ground-truth layout
(`source1_entity_id`, comma-separated `matched_entity_ids`).

In [13]:
with stage('blocking + candidates (test)'):
    display(build_keys('test'))
    build_candidates('test', 'cands_test')
    if USE_EMBEDDING_BLOCKING:
        with stage('embedding candidates (test)'):
            build_embedding_candidates('test', 'cands_test')
            merge_emb_candidates('test', 'cands_test')

kept = []
with stage('features + predict (test)'):
    for batch, X in stream_features('cands_test', 'test', with_label=False):
        keep = pa.array(model.predict_proba(X)[:, 1] >= THRESHOLD)
        kept.append(pa.table({'id1': batch.column('id1'), 'id2': batch.column('id2')}).filter(keep))
matches = pa.concat_tables(kept)
del kept
print(f'predicted pairs: {matches.num_rows:,}')

s1_order = pd.read_csv(SOURCES['test'][1], sep='\t', dtype=str, usecols=['entity_id'])['entity_id'].str.strip()
s1_order = pa.table({'entity_id': s1_order.to_numpy(), 'pos': np.arange(len(s1_order))})
con.register('s1_order', s1_order)
con.register('matches', matches)
submission = con.execute("""
    SELECT s.entity_id AS source1_entity_id,
           coalesce(string_agg(m.id2, ',' ORDER BY m.id2), '') AS matched_entity_ids
    FROM s1_order s LEFT JOIN matches m ON m.id1 = s.entity_id
    GROUP BY s.entity_id, s.pos ORDER BY s.pos
""").df()
con.unregister('s1_order')
con.unregister('matches')
out_path = WORK_DIR / 'test_predictions.tsv'
submission.to_csv(out_path, sep='\t', index=False)
out_path = save_to_cache(out_path)
print(out_path, submission.shape)
n_matches = submission['matched_entity_ids'].map(lambda s: len(s.split(',')) if s else 0)
print(n_matches.value_counts().sort_index().rename('S1 records by number of predicted matches'))
submission.head()

▶ blocking + candidates (test)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

▶ rarest name tokens (test)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✔ features + predict (test): 19,525.3s
predicted pairs: 5,697,270


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/content/drive/MyDrive/er_cache/test_predictions.tsv (1732544, 2)
matched_entity_ids
0     111275
1     162523
2     316637
3     388128
4     336332
5     224229
6     118391
7      50461
8      17561
9       5000
10      1276
11       406
12       150
13        71
14        46
15        25
16        15
17         6
18         2
19         4
20         2
21         1
23         2
26         1
Name: S1 records by number of predicted matches, dtype: int64


,source1_entity_id,matched_entity_ids
0,S1-714132312,"S2-187020300,S3-625880872,S3-867809779"
1,S1-106407869,S3-613056593
2,S1-156285671,"S2-172049742,S2-648730048,S3-339493107,S3-7888..."
3,S1-689823050,"S2-212601842,S2-319486263,S2-65478038,S2-82907..."
4,S1-921369899,"S2-107533354,S2-808920974,S2-995426671,S3-2850..."


### Sanity checks on the submission

In [14]:
print("Input S1 records:", len(s1_order))
print("Submission rows:", len(submission))
print("Unique S1 IDs:", submission["source1_entity_id"].nunique())
print("Duplicate S1 IDs:", submission["source1_entity_id"].duplicated().sum())

Input S1 records: 1732544
Submission rows: 1732544
Unique S1 IDs: 1732544
Duplicate S1 IDs: 0


In [15]:
input_ids = set(s1_order["entity_id"].to_pylist())
output_ids = set(submission["source1_entity_id"])
print("Input IDs:", len(input_ids))
print("Output IDs:", len(output_ids))
print("Missing from output:", len(input_ids - output_ids))
print("Extra in output:", len(output_ids - input_ids))

Input IDs: 1732544
Output IDs: 1732544
Missing from output: 0
Extra in output: 0


In [16]:
print("Total S1 records:", len(submission))
print("\nMatch distribution:")
print(
    submission["matched_entity_ids"]
    .map(lambda s: len(s.split(",")) if s else 0)
    .value_counts()
    .sort_index()
)

Total S1 records: 1732544

Match distribution:
matched_entity_ids
0     111275
1     162523
2     316637
3     388128
4     336332
5     224229
6     118391
7      50461
8      17561
9       5000
10      1276
11       406
12       150
13        71
14        46
15        25
16        15
17         6
18         2
19         4
20         2
21         1
23         2
26         1
Name: count, dtype: int64


In [17]:
zero_matches = submission[submission["matched_entity_ids"].fillna("") == ""]
many_matches = submission[
    submission["matched_entity_ids"].fillna("").map(lambda s: len(s.split(",")) if s else 0) >= 10
]
print("0-match records:", len(zero_matches))
print("10+ match records:", len(many_matches))
display(many_matches.head(20))

0-match records: 111275
10+ match records: 2007


,source1_entity_id,matched_entity_ids
234,S1-17003766,"S2-161594347,S2-341012712,S2-365194406,S2-5589..."
1678,S1-448267460,"S2-17267219,S2-199169193,S2-409585491,S2-75297..."
2613,S1-219817439,"S2-406745137,S2-48067306,S2-559054794,S2-62923..."
3027,S1-971850426,"S2-189249623,S2-404416090,S2-40457265,S2-86880..."
3471,S1-822774741,"S2-409097245,S2-417623756,S2-481802705,S2-6087..."
4014,S1-21237982,"S2-121192935,S2-249827644,S2-403482917,S2-8481..."
5769,S1-206316150,"S2-509751005,S2-628414180,S2-769153246,S2-8652..."
6583,S1-851078815,"S2-474376412,S2-855348473,S2-89200974,S3-26689..."
7580,S1-573862000,"S2-231535678,S2-28158586,S2-283993488,S2-33713..."
8375,S1-665537374,"S2-368720471,S2-779134270,S2-941319128,S3-1195..."


In [18]:
from pathlib import Path
import shutil

src = Path(WORK_DIR) / "test_predictions.tsv"
dst = Path(WORK_DIR) / "matching_results.tsv"

shutil.copy2(src, dst)

print("Created:", dst)
print("Size:", dst.stat().st_size / (1024**2), "MB")

Created: /content/er_work/matching_results.tsv
Size: 91.43263816833496 MB


In [19]:
from google.colab import files

files.download(str(dst))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Where to go from here

1. **Hyperparameter tuning** – wrap the LightGBM fit in an Optuna study (`num_leaves`, `learning_rate`,
   `min_child_samples`, `reg_alpha`, `reg_lambda`) and optimise the F-beta on `cands_val`.
2. **Cross-encoder rerank** – take the top ~2k val/test pairs per S1 above the current threshold and rerank
   them with a multilingual cross-encoder (`xlm-roberta-base` or `bge-reranker-base`). This typically buys
   5–10 F1 on this kind of data.
3. **Embedding blocking in production** – if you enabled `USE_EMBEDDING_BLOCKING`, expect the biggest gains
   on entities whose name tokens are all common and whose address is missing, which is exactly the
   failure mode of lexical-only blocking.
4. **Validate the final `.tsv`** with the competition's `utils/validate_submission.py` before uploading.